<div style="background: linear-gradient(120deg, #1a3a5c 0%, #2d6a9f 60%, #4a9eda 100%); padding: 28px 36px; border-radius: 14px; display: flex; align-items: center; gap: 28px; box-shadow: 0 4px 18px rgba(0,0,0,0.18);">
    <img src='Figures/iteso.jpg' style='height: 110px; border-radius: 8px; background: white; padding: 6px; flex-shrink: 0;'/>
    <div style='border-left: 2px solid rgba(255,255,255,0.4); padding-left: 28px;'>
        <h1 style='margin: 0 0 8px 0; color: white; font-size: 1.5em;'>Ingeniería y Ciencia de Datos</h1>
        <h3 style='margin: 0 0 8px 0; color: white; font-size: 1.5em;'>Laboratorio de Procesamiento de Datos</h3>
        <h3 style='margin: 0; color: rgba(255,255,255,0.8); font-weight: normal; font-size: 1.05em;'>Módulo 2: Generación y transformación de datos</h3>
    </div>
</div>


## Enriquecimiento y creación de variables

Transformar datos no consiste únicamente en corregir errores o cambiar formatos. 

> También podemos **enriquecerlos**: agregar información útil o crear nuevas variables a partir de los datos disponibles. Por ejemplo, una fecha de nacimiento permite calcular una edad, una cantidad de visitas puede convertirse en una categoría (baja, media o alta), y una medida puede convertirse a otra unidad o escalarse.

Las variables derivadas pueden facilitar el análisis, apoyar reportes de **Business Intelligence (BI)** y preparar los datos para estudios posteriores o, más adelante, para modelos de inteligencia artificial. 

>Que una variable se pueda calcular no significa automáticamente que sea útil: debe responder a una pregunta y su definición debe poder explicarse.

**No existe una secuencia universal de generación de nuevos datos que se aplique igual a todos los proyectos.** Sin embargo, podemos usar un **proceso de trabajo** para tomar decisiones ordenadas, revisar los resultados y documentar cómo se crearon las variables. 
- Este proceso es iterativo: al validar el resultado podríamos necesitar ajustar la pregunta o la transformación.

1. **Definir el propósito.** ¿Qué pregunta queremos responder o qué decisión apoyará el análisis? Esto ayuda a elegir qué información realmente aporta valor.
2. **Revisar los datos disponibles.** Identificar columnas, tipos, unidades, valores faltantes, rangos y posibles inconsistencias. Confirmar qué significa cada campo con el contexto del problema.
3. **Elegir una estrategia de enriquecimiento.** Por ejemplo: calcular una variable (edad o importe total), agrupar valores en categorías, convertir unidades, escalar una medida o combinar la tabla con otra fuente confiable.
4. **Aplicar y nombrar las transformaciones.** Crear columnas nuevas con fórmulas claras y nombres descriptivos. Cuando sea útil, conservar las columnas originales para poder comparar y rastrear el cálculo.
5. **Validar el resultado.** Comprobar algunos cálculos manualmente, revisar tipos y rangos, contar valores faltantes y verificar que una unión de tablas no haya duplicado ni eliminado registros inesperadamente.
6. **Documentar y mantener.** Registrar fuente, fórmula, unidades, supuestos, fecha de referencia y limitaciones. Si los datos de origen cambian, definir cómo y cuándo se actualizan las variables.

```mermaid
flowchart TD
    A[Definir el propósito o pregunta] --> B[Revisar datos y contexto]
    B --> C[Elegir transformación o fuente adicional]
    C --> D[Crear variables, combinar o resumir]
    D --> E[Validar resultados]
    E --> F{¿Son correctos y útiles?}
    F -- Sí --> G[Documentar y utilizar]
    F -- No --> H[Revisar propósito, datos o regla]
    H --> B
```


In [4]:
import os
import pandas as pd
import matplotlib.pyplot as plt

carpeta_datos = 'Data'

## 1. Ejemplo Dummy

Usaremos algunos registros **sintéticos** de clientes y una tabla pequeña de actividad diaria de una cafetería. Sintético significa que los datos son inventados para practicar; no describen clientes ni un negocio reales.

En proyectos reales, las fechas, ventas y demás datos se obtendrían de los sistemas de la organización. La práctica se concentra en cómo enriquecerlos.

In [5]:
clientes = pd.DataFrame({
    'cliente_id': ['C001', 'C002', 'C003', 'C004', 'C005'],
    'fecha_nacimiento': pd.to_datetime([
        '2008-04-12', '1998-11-03', '1985-06-25',
        '1962-02-18', '2001-10-20'
    ])
})

clientes

,cliente_id,fecha_nacimiento
0,C001,2008-04-12
1,C002,1998-11-03
2,C003,1985-06-25
3,C004,1962-02-18
4,C005,2001-10-20


### 1.2. Crear una nueva variable

Crear la variable edad apartir de la fecha: La edad no siempre viene guardada en una tabla, pero puede calcularse a partir de la fecha de nacimiento. Primero elegimos una **fecha de referencia**. Para este ejemplo será el 22 de octubre de 2026, usar una fecha fija hace que el ejercicio noa de el mismo resultado cuando se vuelva a ejecutar.

No basta con restar los años: si la persona todavía no ha cumplido años en la fecha de referencia, debemos restar uno. La comparación del mes y día permite considerar ese detalle. 

- En pandas, `.dt.year`, `.dt.month` y `.dt.day` extraen esas partes de una columna de fechas. 

La edad calculada será una **variable derivada**

In [6]:
clientes['fecha_nacimiento']

0   2008-04-12
1   1998-11-03
2   1985-06-25
3   1962-02-18
4   2001-10-20
Name: fecha_nacimiento, dtype: datetime64[us]

In [7]:
fecha_corte = pd.Timestamp('2026-10-20')
cumple_anos_pendiente = (
    (clientes['fecha_nacimiento'].dt.month > fecha_corte.month)
    | ((clientes['fecha_nacimiento'].dt.month == fecha_corte.month)& (clientes['fecha_nacimiento'].dt.day > fecha_corte.day))
)


In [4]:
cumple_anos_pendiente

0    False
1     True
2    False
3    False
4    False
Name: fecha_nacimiento, dtype: bool

In [ ]:
clientes['edad'] = (
    fecha_corte.year
    - clientes['fecha_nacimiento'].dt.year
    - cumple_anos_pendiente.astype(int) #Sirve para comparar 
)

clientes

,cliente_id,fecha_nacimiento,edad
0,C001,2008-04-12,18
1,C002,1998-11-03,27
2,C003,1985-06-25,41
3,C004,1962-02-18,64
4,C005,2001-10-20,25


### 1.3. Agrupar valores en categorías

A veces es más fácil resumir o comunicar una cantidad mediante grupos. `pd.cut` divide una columna numérica en intervalos. En este ejemplo agruparemos las edades, los límites son decisiones que deben elegirse según el contexto, no reglas universales. Por defecto, cada intervalo incluye su límite derecho, `include_lowest=True` también incluye el valor más bajo del primer intervalo.

In [9]:
limites_edad = [0, 17, 30, 60, 120]
etiquetas_edad = ['0-17', '18-30', '31-60', '61 o más']

clientes['grupo_edad'] = pd.cut(
    clientes['edad'],
    bins=limites_edad,
    labels=etiquetas_edad,
    include_lowest=True
)

display(clientes)
print('Cantidad de clientes en cada grupo:')
display(clientes['grupo_edad'].value_counts(sort=False))

,cliente_id,fecha_nacimiento,edad,grupo_edad
0,C001,2008-04-12,18,18-30
1,C002,1998-11-03,27,18-30
2,C003,1985-06-25,41,31-60
3,C004,1962-02-18,64,61 o más
4,C005,2001-10-20,25,18-30


Cantidad de clientes en cada grupo:


grupo_edad
0-17        0
18-30       3
31-60       1
61 o más    1
Name: count, dtype: int64

## 2. Dataset: pingüinos de Palmer

Ahora aplicaremos las mismas ideas a un conjunto de datos publicado en la web. **Palmer Penguins** contiene mediciones de pingüinos observados en tres islas del archipiélago Palmer, en la Antártida. El conjunto tiene **344** registros e incluye especie, isla, longitud y profundidad del pico, longitud de la aleta, masa corporal, sexo y año.

Fuentes: [repositorio Palmer Penguins](https://github.com/allisonhorst/palmerpenguins), [archivo CSV utilizado](https://raw.githubusercontent.com/allisonhorst/palmerpenguins/main/inst/extdata/penguins.csv).

Primero revisaremos las columnas y los valores faltantes. 


In [11]:
pinguinos = pd.read_csv(os.path.join(carpeta_datos, 'penguins.csv'))


In [15]:
pinguinos.describe()

,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,year
count,342.000000,342.000000,342.000000,342.000000,344.000000
mean,43.921930,17.151170,200.915205,4201.754386,2008.029070
std,5.459584,1.974793,14.061714,801.954536,0.818356
min,32.100000,13.100000,172.000000,2700.000000,2007.000000
25%,39.225000,15.600000,190.000000,3550.000000,2007.000000
50%,44.450000,17.300000,197.000000,4050.000000,2008.000000
75%,48.500000,18.700000,213.000000,4750.000000,2009.000000
max,59.600000,21.500000,231.000000,6300.000000,2009.000000


In [14]:
pinguinos.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007


In [16]:
pinguinos.isna().sum()

species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
year                  0
dtype: int64

**Vamos a crear columnas que podrían facilitar la lectura y el resumen de los datos:**

- `body_mass_kg`: convierte la masa de gramos a kilogramos (`gramos / 1000`).
- `flipper_length_cm`: convierte la longitud de la aleta de milímetros a centímetros (`milímetros / 10`).
- `bill_length_depth_ratio`: divide la longitud del pico entre su profundidad. Es un **índice calculado**.
- `bill_length_group`: agruparemos la longitud del pico en intervalos ilustrativos. 
- `body_mass_zscore`: expresa la masa en relación con el promedio de esta muestra. Un valor positivo está por encima del promedio y uno negativo, por debajo.

In [18]:
pinguinos_enriquecidos = pinguinos.copy()

In [21]:
#Nuevas columnas
pinguinos_enriquecidos['body_mass_kg'] = pinguinos_enriquecidos['body_mass_g'] / 1000
pinguinos_enriquecidos['body_mass_kg']

0      3.750
1      3.800
2      3.250
3        NaN
4      3.450
       ...  
339    4.000
340    3.400
341    3.775
342    4.100
343    3.775
Name: body_mass_kg, Length: 344, dtype: float64

In [ ]:
#pd.cut

In [ ]:
#Columnas estadisticas

In [ ]:
#Valores faltantes

Además de crear columnas para cada pingüino, podemos **generar una nueva tabla** que resuma la información por especie e isla. `groupby` reúne las filas de cada combinación y `agg` calcula medidas para cada grupo. Esta tabla podría servir como punto de partida para un reporte de BI.

In [ ]:
#Reporte con algunas métricas
resumen_pinguinos = None


Generar algunas gráficas

## 3. Enriquecer datos de alquiler de bicicletas

En este ejemplo trabajaremos con el conjunto **Bike Sharing Dataset** (`hour.csv`) extraído del repositorio [archivo original de UCI](https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip). La descripción original de las columnas está en `Data/bike_sharing_Readme.txt`. Este dataset contiene registros reales por hora del sistema **Capital Bikeshare** en Washington, D. C., durante 2011 y 2012, junto con información de calendario y clima. Cada fila representa una hora e incluye, entre otras columnas, fecha, estación, clima, temperatura normalizada, humedad, velocidad del viento y cantidad de bicicletas rentadas.

**¿Cómo podemos preparar estos registros para explorar la demanda por hora, periodo del día, tipo de día, estación del año y condiciones meteorológicas?**

- Para esto vamos a crear variables más interpretables que aporten valor para un análisis más comprensible.

In [ ]:
bicicletas_original = pd.read_csv(os.path.join(carpeta_datos, 'hour.csv'))


Algunas columnas están codificadas como números, pero representan categorías. Según la [descripción de UCI](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset):

- `season`: 1 invierno, 2 primavera, 3 verano, 4 otoño.
- `weathersit`: 1 despejado o parcialmente nublado, 2 neblina o nubes, 3 nieve o lluvia ligera, 4 condiciones severas.
- `temp` está normalizada entre 0 y 1 a partir de una temperatura en °C entre -8 y 39. `atemp` usa una escala distinta, entre -16 y 50 °C.
- `hum` es humedad dividida entre 100.
- `casual` y `registered` son conteos de usuarios, `cnt` es el total de alquileres.

No debemos tratar el código `season=1` como una medida numérica: el número identifica una categoría. Usaremos etiquetas para que las tablas sean más legibles.

In [ ]:
columnas_conteos = ['casual', 'registered', 'cnt']
diferencia_conteos = (
    bicicletas_original['casual']
    + bicicletas_original['registered']
    - bicicletas_original['cnt']
)

print('Filas duplicadas por fecha y hora:', bicicletas_original.duplicated(['dteday', 'hr']).sum())
print('Filas donde casual + registered no coincide con cnt:', (diferencia_conteos != 0).sum())
print('\nRangos de algunas columnas:')
bicicletas_original[columnas_conteos + ['temp', 'atemp', 'hum', 'windspeed']].describe().round(2)

A partir de la fecha y la hora construiremos el momento completo (`fecha_hora`), el nombre del día, si corresponde a fin de semana y un periodo del día. También convertiremos las escalas normalizadas a sus unidades descritas por UCI y traduciremos los códigos de estación y clima.

Crearemos `periodo_dia`donde: madrugada (00–05), mañana (06–11), tarde (12–17) y noche (18–23). No son una clasificación universal es una elección que hacemos. La columna original se conserva y las variables derivadas se agregan a una copia.

In [ ]:
bicicletas = bicicletas_original.copy()
bicicletas['fecha'] = pd.to_datetime(bicicletas['dteday'])
bicicletas['fecha_hora'] = bicicletas['fecha'] + pd.to_timedelta(bicicletas['hr'], unit='h')
bicicletas['nombre_dia'] = bicicletas['fecha'].dt.day_name()
bicicletas['es_fin_de_semana'] = bicicletas['fecha'].dt.dayofweek.isin([5, 6])

bicicletas['periodo_dia'] = pd.cut(
    bicicletas['hr'],
    bins=[0, 6, 12, 18, 24],
    labels=['Madrugada', 'Mañana', 'Tarde', 'Noche'],
    right=False,
    include_lowest=True
)


In [ ]:
etiquetas_estacion = {
    1: 'Invierno', 2: 'Primavera', 3: 'Verano', 4: 'Otoño'
}
etiquetas_clima = {
    1: 'Despejado o parcialmente nublado',
    2: 'Neblina o nublado',
    3: 'Nieve o lluvia ligera',
    4: 'Condiciones severas'
}

In [ ]:
bicicletas['estacion_nombre'] = bicicletas['season'].map(etiquetas_estacion)
bicicletas['clima_nombre'] = bicicletas['weathersit'].map(etiquetas_clima)

#Conversion a grados c
bicicletas['temperatura_c'] = bicicletas['temp'] * (39 - (-8)) + (-8)
bicicletas['sensacion_termica_c'] = bicicletas['atemp'] * (50 - (-16)) + (-16)

bicicletas['humedad_pct'] = bicicletas['hum'] * 100

bicicletas['proporcion_usuarios_registrados'] = (
    bicicletas['registered'] / bicicletas['cnt'].where(bicicletas['cnt'] != 0)
)


In [ ]:

columnas_nuevas = [
    'fecha_hora', 'nombre_dia', 'es_fin_de_semana', 'periodo_dia',
    'estacion_nombre', 'clima_nombre', 'temperatura_c',
    'sensacion_termica_c', 'humedad_pct', 'proporcion_usuarios_registrados'
]
bicicletas[['dteday', 'hr', 'season', 'weathersit', 'cnt'] + columnas_nuevas].head(12).round(2)

**Una variable derivada puede enriquecer cada fila** una **tabla agregada** resume varias filas. Aquí generaremos dos productos sencillos: la demanda promedio por hora y una tabla diaria. 

- `mean` contesta cuánto se renta en una hora típica de ese grupo
- `sum` permite calcular el total observado en el día. 

In [ ]:
demanda_por_hora = (
    bicicletas
    .groupby('hr', as_index=False, observed=True)
    .agg(
        alquileres_promedio=('cnt', 'mean'),
        casuales_promedio=('casual', 'mean'),
        registrados_promedio=('registered', 'mean')
    )
)


In [ ]:
demanda_por_periodo = (
    bicicletas
    .groupby(['es_fin_de_semana', 'periodo_dia'], as_index=False, observed=True)
    .agg(
        horas_observadas=('cnt', 'size'),
        alquileres_promedio=('cnt', 'mean'),
        humedad_promedio_pct=('humedad_pct', 'mean')
    )
)


In [ ]:
resumen_diario = (
    bicicletas
    .groupby('fecha', as_index=False)
    .agg(
        alquileres_total=('cnt', 'sum'),
        usuarios_casuales_total=('casual', 'sum'),
        usuarios_registrados_total=('registered', 'sum'),
        horas_con_registro=('hr', 'nunique'),
        temperatura_promedio_c=('temperatura_c', 'mean')
    )
)


In [ ]:
print('Promedio de alquileres por hora del día:')
display(demanda_por_hora.round(1))

In [ ]:
print('Comparación por fin de semana y periodo:')
display(demanda_por_periodo.round(1))

In [ ]:
print('Primeros registros del resumen diario:')
display(resumen_diario.head().round(1))

Las tablas permiten explorar cuándo se registraron más alquileres y comparar periodos o días. Son **descripciones de estos datos históricos**. Una diferencia entre días laborables y fines de semana no demuestra por sí sola que el tipo de día sea la causa: también pueden influir clima, eventos, disponibilidad de bicicletas y otros factores.


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(demanda_por_hora['hr'], demanda_por_hora['alquileres_promedio'], marker='o')
plt.title('Alquileres promedio por hora (2011–2012)')
plt.xlabel('Hora del día')
plt.ylabel('Alquileres promedio por registro horario')
plt.xticks(range(0, 24, 2))
plt.grid(alpha=0.3)
plt.show()

print('Resumen diario, en días con más alquileres:')
display(resumen_diario.nlargest(5, 'alquileres_total').round(1))

## 4. Enriquecer registros con contexto geográfico y demográfico

Leeremos las tres respuestas JSON del Banco Mundial guardadas en `Data`. Conservamos las respuestas originales para mantener los metadatos de las fuentes.

Un negocio puede tener registros de actividad asociados a un país o región y querer agregar contexto para un reporte. En lugar de usar direcciones de personas, aquí trabajaremos con **códigos ISO de país** y datos públicos agregados del Banco Mundial. 

- Haremos una unión mediante una clave geográfica común, igual que se podría relacionar un registro con una tabla de ciudades, estados o códigos postales cuando el nivel y el uso sean apropiados.

- Usaremos indicadores del año **2023**: población total (`SP.POP.TOTL`), PIB per cápita en dólares corrientes (`NY.GDP.PCAP.CD`) y región/nivel de ingreso del catálogo de países. Estos valores describen países o economías.

La tabla de compras y sus importes son datos dummys creados para este ejercicio. La fuente real que agregamos es la información geográfica y económica del Banco Mundial. 

Fuentes: [API de países del Banco Mundial](https://api.worldbank.org/v2/country?format=json&per_page=300), [indicador de población](https://api.worldbank.org/v2/country/all/indicator/SP.POP.TOTL?date=2023&format=json&per_page=300) e [indicador de PIB per cápita](https://api.worldbank.org/v2/country/all/indicator/NY.GDP.PCAP.CD?date=2023&format=json&per_page=300).

In [ ]:
import json

def leer_datos_banco_mundial(archivo):
    with open(archivo, encoding='utf-8') as entrada:
        contenido = json.load(entrada)
    if len(contenido) < 2 or contenido[1] is None:
        raise ValueError(f'El archivo no contiene datos del Banco Mundial: {archivo}')
    return contenido[1]

paises_api = leer_datos_banco_mundial(
    os.path.join(carpeta_datos, 'banco_mundial_paises.json')
)
poblacion_api = leer_datos_banco_mundial(
    os.path.join(carpeta_datos, 'banco_mundial_poblacion_2023.json')
)
pib_api = leer_datos_banco_mundial(
    os.path.join(carpeta_datos, 'banco_mundial_pib_per_capita_2023.json')
)

print('Registros de países recibidos:', len(paises_api))
print('Observaciones de población recibidas:', len(poblacion_api))
print('Observaciones de PIB per cápita recibidas:', len(pib_api))

### Preparar las tablas y unirlas por el código del país

Los archivos locales conservan las respuestas originales de la API en formato JSON, incluidos sus metadatos. Convertiremos cada respuesta a una tabla de pandas y dejaremos solo columnas necesarias. 

- El código de tres letras (por ejemplo, `MEX`) será nuestra **clave de unión**. Excluimos los agregados regionales del Banco Mundial porque el ejemplo asigna contexto a países individuales.

In [ ]:
tabla_paises = pd.DataFrame([
    {
        'pais_iso3': pais['id'],
        'pais': pais['name'],
        'region': pais['region']['value'].strip(),
        'nivel_ingreso_pais': pais['incomeLevel']['value'].strip(),
        'codigo_nivel_ingreso': pais['incomeLevel']['id']
    }
    for pais in paises_api
    if pais['incomeLevel']['id'] in {'LIC', 'LMC', 'UMC', 'HIC'}
])

codigos_pais = set(tabla_paises['pais_iso3'])

tabla_poblacion = pd.DataFrame([
    {'pais_iso3': dato['countryiso3code'], 'poblacion_2023': dato['value']}
    for dato in poblacion_api
    if dato['value'] is not None and dato['countryiso3code'] in codigos_pais
])

tabla_pib = pd.DataFrame([
    {'pais_iso3': dato['countryiso3code'], 'pib_per_capita_usd_2023': dato['value']}
    for dato in pib_api
    if dato['value'] is not None and dato['countryiso3code'] in codigos_pais
])

contexto_paises = (
    tabla_paises
    .merge(tabla_poblacion, on='pais_iso3', how='left', validate='one_to_one')
    .merge(tabla_pib, on='pais_iso3', how='left', validate='one_to_one')
)

display(contexto_paises[contexto_paises['pais_iso3'].isin(['MEX', 'USA', 'COL', 'BRA'])].round(2))

### Agregar contexto a una tabla de actividad

Ahora relacionamos cada registro con la tabla geográfica mediante `pais_iso3`. En un proyecto real, esa clave podría provenir de un país seleccionado por el usuario o de una dirección validada y convertida a una región con permisos adecuados. 

In [ ]:
compras_demo = pd.DataFrame({
    'registro_id': ['R001', 'R002', 'R003', 'R004', 'R005', 'R006'],
    'pais_iso3': ['MEX', 'USA', 'COL', 'BRA', 'MEX', 'USA'],
    'importe_compra_usd': [35.50, 82.00, 24.75, 51.20, 63.00, 45.00]
})

compras_enriquecidas = compras_demo.merge(
    contexto_paises,
    on='pais_iso3',
    how='left',
    validate='many_to_one',
    indicator=True
)

print('Registros antes de la unión:', len(compras_demo))
print('Registros después de la unión:', len(compras_enriquecidas))
print('Resultado de la unión:')
display(compras_enriquecidas[[
    'registro_id', 'pais_iso3', 'pais', 'region', 'nivel_ingreso_pais',
    'poblacion_2023', 'pib_per_capita_usd_2023', 'importe_compra_usd', '_merge'
]].round(2))

print('¿Algún código no encontró país?', (compras_enriquecidas['_merge'] != 'both').any())


**Idea clave:** enriquecer datos es crear variables útiles con reglas comprensibles, justificadas y documentadas.

## Práctica de Laboratorio
Para el dataset **Palmer Penguins**
1. Propón otra variable derivada que pudiera ayudar en un reporte y escribe en una frase cómo se calcularía y qué limitación tendría.

Usando el datset **Bike Sharing**, realizar lo siguiente:
1. Cambia los intervalos de `periodo_dia` y explica cómo cambia la tabla `demanda_por_periodo`.
2. Compara la cantidad promedio de usuarios casuales y registrados en `demanda_por_hora`. ¿En qué horas difieren más?
3. Crea una tabla que resuma los alquileres promedio por `estacion_nombre` y `clima_nombre`. Recuerda incluir el número de observaciones de cada grupo.
5. Propón una variable derivada adicional para planear la operación de bicicletas. Describe su fórmula, unidades y una limitación.

Usando las tres respuestas **JSON del Banco Mundial** y el datset **compras_demo**, realizar lo siguiente:

1. Cambia uno de los códigos `pais_iso3` de `compras_demo` por otro país que aparezca en `contexto_paises`. ¿Qué columnas se agregan?
2. Prueba un código que no exista y observa `_merge`. ¿Por qué es útil verificar si todas las filas encontraron correspondencia?

In [22]:
import os

nombre_salida = "repomix-output.txt"
extensiones_validas = {".py", ".ipynb", ".md"}
carpetas_ignorar = {".venv", ".git", "__pycache__", "salidas", "datos", "Figures"}

with open(nombre_salida, "w", encoding="utf-8") as salida:
    for raiz, directorios, archivos in os.walk("."):
        directorios[:] = [d for d in directorios if d not in carpetas_ignorar]
        for archivo in archivos:
            ext = os.path.splitext(archivo)[1].lower()
            if ext in extensiones_validas and archivo != nombre_salida:
                ruta = os.path.join(raiz, archivo)
                salida.write(f"\n{'='*70}\nARCHIVO: {ruta}\n{'='*70}\n\n")
                try:
                    with open(ruta, "r", encoding="utf-8", errors="ignore") as f:
                        salida.write(f.read())
                except Exception as e:
                    salida.write(f"Error al leer {ruta}: {e}\n")

print(f"Archivo generado en: {os.path.abspath(nombre_salida)}")

Archivo generado en: c:\Users\santi\OneDrive - ITESO\ITESO\Semestre 3\Procesamiento de Datos\Laboratorio_de_Procesamiento_de_Datos_02026\Modulo II\repomix-output.txt
